In [ ]:
# If needed, run once: %pip install -q tensorflow scikit-learn pillow matplotlib scipy
import random, os
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.mixture import GaussianMixture
import joblib

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print("TensorFlow:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))

## Cell 2 — Mount Drive and locate inputs

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DATA_DIR = Path("/content/drive/MyDrive/CetaceanWhistleAI")
if not DATA_DIR.exists():
    raise FileNotFoundError(f"Folder not found: {DATA_DIR}")

EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}
image_paths = sorted(
    [p for p in DATA_DIR.iterdir() if p.is_file() and p.suffix.lower() in EXTS],
    key=lambda p: p.name.lower()
)
if not image_paths:
    raise FileNotFoundError(f"No image files found in {DATA_DIR}")
print("Dataset folder:", DATA_DIR)
print("Images:", [p.name for p in image_paths])

## Cell 3 — Load and normalize images
Default preprocessing does not crop, because cropping can cut off a contour. Inspect Cell 4 before changing this.

In [ ]:
IMAGE_SIZE = (128, 128)  # height, width

def load_image(path):
    im = Image.open(path).convert("L")
    im = im.resize((IMAGE_SIZE[1], IMAGE_SIZE[0]), Image.Resampling.LANCZOS)
    return (np.asarray(im, dtype=np.float32) / 255.0)[..., None]

images = np.stack([load_image(p) for p in image_paths]).astype(np.float32)
assert images.ndim == 4 and images.shape[1:] == (128, 128, 1), images.shape
print("Loaded images:", images.shape)
print("Pixel range:", float(images.min()), "to", float(images.max()))
print("Mean/std:", float(images.mean()), float(images.std()))

## Cell 4 — Inspect inputs and create batches

In [ ]:
n_images = len(images)
cols = min(7, n_images)
rows = int(np.ceil(n_images / cols))
fig, axes = plt.subplots(rows, cols, figsize=(2.3*cols, 2.5*rows))
axes = np.array(axes).reshape(-1)
for i, ax in enumerate(axes):
    if i < n_images:
        ax.imshow(images[i, :, :, 0], cmap="gray", vmin=0, vmax=1)
        ax.set_title(image_paths[i].name, fontsize=9)
    ax.axis("off")
plt.suptitle("Prepared grayscale whistle-contour inputs")
plt.tight_layout(); plt.show()

BATCH_SIZE = min(4, n_images)
train_ds = (tf.data.Dataset.from_tensor_slices(images)
            .shuffle(n_images, seed=SEED, reshuffle_each_iteration=True)
            .batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE))
print("Images:", n_images, "| batch size:", BATCH_SIZE,
      "| batches:", int(np.ceil(n_images / BATCH_SIZE)))

## Cell 5 — Build encoder and decoder
The encoder outputs latent mean and log-variance; the decoder returns a 128×128 grayscale image.

In [ ]:
LATENT_DIM = 64

def build_encoder(latent_dim=LATENT_DIM):
    inp = keras.Input((128, 128, 1), name="whistle_image")
    x = layers.Conv2D(32, 3, 2, padding="same", activation="swish")(inp)
    x = layers.Conv2D(32, 3, 2, padding="same", activation="swish")(x)
    x = layers.Conv2D(64, 3, 2, padding="same", activation="swish")(x)
    x = layers.Conv2D(64, 3, 2, padding="same", activation="swish")(x)
    x = layers.Flatten()(x)
    return keras.Model(inp, [layers.Dense(latent_dim, name="z_mean")(x),
                             layers.Dense(latent_dim, name="z_log_var")(x)],
                       name="vae_encoder")

def build_decoder(latent_dim=LATENT_DIM):
    inp = keras.Input((latent_dim,), name="latent_vector")
    x = layers.Dense(8*8*64, activation="swish")(inp)
    x = layers.Reshape((8, 8, 64))(x)
    x = layers.Conv2DTranspose(64, 3, 2, padding="same", activation="swish")(x)
    x = layers.Conv2DTranspose(64, 3, 2, padding="same", activation="swish")(x)
    x = layers.Conv2DTranspose(32, 3, 2, padding="same", activation="swish")(x)
    x = layers.Conv2DTranspose(32, 3, 2, padding="same", activation="swish")(x)
    out = layers.Conv2D(1, 3, padding="same", activation="sigmoid",
                        name="reconstructed_image")(x)
    return keras.Model(inp, out, name="vae_decoder")

encoder = build_encoder()
decoder = build_decoder()
encoder.summary()
decoder.summary()

## Cell 6 — Sampling and contour-preserving loss
This experimental loss combines binary cross-entropy, Sobel-edge difference and KL divergence. It is not claimed to be the paper's exact loss.

In [ ]:
BETA_KL = 1e-5
EDGE_WEIGHT = 0.5

def reparameterize(z_mean, z_log_var):
    eps = tf.random.normal(tf.shape(z_mean))
    return z_mean + tf.exp(0.5 * z_log_var) * eps

def vae_losses(real, recon, z_mean, z_log_var):
    bce = tf.reduce_mean(keras.losses.binary_crossentropy(real, recon))
    real_edges = tf.image.sobel_edges(real)
    recon_edges = tf.image.sobel_edges(recon)
    edge = tf.reduce_mean(tf.abs(real_edges - recon_edges))
    kl = tf.reduce_mean(tf.reduce_sum(
        -0.5 * (1.0 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var)),
        axis=1
    ))
    total = bce + EDGE_WEIGHT * edge + BETA_KL * kl
    return total, bce, edge, kl

print("Loss functions ready.")

## Cell 7 — Train the VAE
Run Cell 5 first for a fresh model. Rerunning this cell alone continues training the current weights.

In [ ]:
EPOCHS = 300
optimizer = keras.optimizers.Adam(learning_rate=1e-4, clipnorm=1.0)
history = {"total": [], "reconstruction": [], "edge": [], "kl": []}

@tf.function
def train_step(real):
    with tf.GradientTape() as tape:
        mu, logvar = encoder(real, training=True)
        z = reparameterize(mu, logvar)
        recon = decoder(z, training=True)
        losses = vae_losses(real, recon, mu, logvar)
    variables = encoder.trainable_variables + decoder.trainable_variables
    grads = tape.gradient(losses[0], variables)
    optimizer.apply_gradients([(g, v) for g, v in zip(grads, variables) if g is not None])
    return losses

for epoch in range(EPOCHS):
    batches = []
    for batch in train_ds:
        batches.append([float(x.numpy()) for x in train_step(batch)])
    avg = np.mean(batches, axis=0)
    for key, value in zip(history.keys(), avg):
        history[key].append(float(value))
    if epoch == 0 or (epoch + 1) % 25 == 0:
        print(f"Epoch {epoch+1:03d}/{EPOCHS} | total={avg[0]:.5f} | "
              f"BCE={avg[1]:.5f} | edge={avg[2]:.5f} | KL={avg[3]:.3f}")

## Cell 8 — Plot losses

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, key, title in zip(
    axes,
    ["total", "reconstruction", "edge"],
    ["Total VAE loss", "Reconstruction BCE", "Contour edge loss"]
):
    ax.plot(history[key]); ax.set_title(title)
    ax.set_xlabel("Epoch"); ax.set_ylabel("Loss"); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## Cell 9 — Evaluate and visualize reconstructions
Uses the latent mean for deterministic reconstructions.

In [ ]:
z_mean_all, z_log_var_all = encoder(images, training=False)
reconstructed = np.clip(decoder(z_mean_all, training=False).numpy(), 0, 1)
mse = float(np.mean((images - reconstructed) ** 2))
edge_mae = float(tf.reduce_mean(tf.abs(
    tf.image.sobel_edges(images) - tf.image.sobel_edges(reconstructed)
)).numpy())
print("Input shape:", images.shape, "| reconstruction shape:", reconstructed.shape)
print("Input min/max:", float(images.min()), float(images.max()))
print("Output min/max:", float(reconstructed.min()), float(reconstructed.max()))
print("Input mean/std:", float(images.mean()), float(images.std()))
print("Output mean/std:", float(reconstructed.mean()), float(reconstructed.std()))
print("Reconstruction MSE:", mse, "| contour edge MAE:", edge_mae)

fig, axes = plt.subplots(2, n_images, figsize=(2.1*n_images, 4.5))
if n_images == 1: axes = axes.reshape(2, 1)
for i in range(n_images):
    axes[0, i].imshow(images[i, :, :, 0], cmap="gray", vmin=0, vmax=1)
    axes[0, i].set_title(f"Input {i+1}"); axes[0, i].axis("off")
    axes[1, i].imshow(reconstructed[i, :, :, 0], cmap="gray", vmin=0, vmax=1)
    axes[1, i].set_title(f"Reconstruction {i+1}"); axes[1, i].axis("off")
plt.tight_layout(); plt.show()

## Cell 10 — Fit a regularized GMM to latent means
GMM convergence only means fitting terminated; it does not prove the clusters are biologically meaningful.

In [ ]:
latent_means = z_mean_all.numpy()
gmm = GaussianMixture(
    n_components=min(2, len(latent_means)),
    covariance_type="diag",
    n_init=5,
    max_iter=500,
    reg_covar=1e-3,
    random_state=SEED
)
gmm.fit(latent_means)
print("Latent means:", latent_means.shape)
print("Converged:", gmm.converged_, "| iterations:", gmm.n_iter_,
      "| components:", gmm.n_components)

## Cell 11 — Generate samples from the GMM

In [ ]:
def generate_gmm_whistles(n_samples=8):
    z, component_ids = gmm.sample(n_samples)
    generated = decoder.predict(z.astype(np.float32), verbose=0)
    return np.clip(generated, 0, 1), component_ids

generated_images, component_ids = generate_gmm_whistles(8)
print("Generated shape:", generated_images.shape)
print("Sampled components:", component_ids)
fig, axes = plt.subplots(1, len(generated_images), figsize=(16, 3))
for i, ax in enumerate(axes):
    ax.imshow(generated_images[i, :, :, 0], cmap="gray", vmin=0, vmax=1,
              interpolation="nearest")
    ax.set_title(f"Generated {i+1}"); ax.axis("off")
plt.tight_layout(); plt.show()

## Cell 12 — Optional local latent variations
Samples near learned latent means. This can test local decoder behavior, but it is not proof of realistic new whistles.

In [ ]:
def generate_local_whistles(n_samples=8, variation=0.10, random_state=SEED):
    means = z_mean_all.numpy()
    scale = np.maximum(np.std(means, axis=0), 1e-3)
    rng = np.random.default_rng(random_state)
    indices = rng.choice(len(means), size=n_samples, replace=True)
    z = means[indices] + variation * scale[None, :] * rng.normal(
        size=(n_samples, LATENT_DIM)
    )
    return np.clip(decoder.predict(z.astype(np.float32), verbose=0), 0, 1)

local_samples = generate_local_whistles(8, variation=0.10)
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for i, ax in enumerate(axes.flat):
    ax.imshow(local_samples[i, :, :, 0], cmap="gray", vmin=0, vmax=1,
              interpolation="nearest")
    ax.set_title(f"Local sample {i+1}"); ax.axis("off")
plt.tight_layout(); plt.show()

## Cell 13 — Simple generated-image diversity diagnostic

In [ ]:
def report_diversity(samples):
    flat = samples.reshape(len(samples), -1)
    variance = float(np.mean(np.var(flat, axis=0)))
    d = [float(np.mean((samples[i] - samples[j])**2))
         for i in range(len(samples)) for j in range(i+1, len(samples))]
    print("Samples:", len(samples))
    print("Mean per-pixel variance:", variance)
    print("Mean pairwise MSE:", float(np.mean(d)) if d else float("nan"))

report_diversity(generated_images)
report_diversity(local_samples)

## Cell 14 — Save models and artifacts

In [ ]:
OUTPUT_DIR = Path("/content/cetacean_vae_gmm_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
encoder.save(OUTPUT_DIR / "encoder_contour.keras")
decoder.save(OUTPUT_DIR / "decoder_contour.keras")
joblib.dump(gmm, OUTPUT_DIR / "latent_gmm_contour.joblib")
np.save(OUTPUT_DIR / "preprocessed_images.npy", images)
np.save(OUTPUT_DIR / "latent_means.npy", latent_means)
np.save(OUTPUT_DIR / "reconstructed_images.npy", reconstructed)
np.save(OUTPUT_DIR / "generated_gmm_images.npy", generated_images)
np.save(OUTPUT_DIR / "generated_local_images.npy", local_samples)
np.save(OUTPUT_DIR / "training_history.npy", history, allow_pickle=True)
print("Saved artifacts to:", OUTPUT_DIR)
print("Files:", sorted(p.name for p in OUTPUT_DIR.iterdir()))

## Run instructions and limitations

1. Upload this notebook to Google Colab.
2. Run **Cell 1 through Cell 14 in order**.
3. Inspect Cell 4 to ensure inputs are not cropped or distorted.
4. Inspect Cell 9 before trusting generated samples.
5. This is the VAE+GMM diagnostic version. It does not include WGAN-GP and is not verified against the paper. The original paper is needed to make the implementation paper-faithful.